# Module 4: Tool returns that tell the truth

> **Prototype notebook.** It walks this module's steps with the evidence kept on the page and links
> straight to the right place in MLflow and the OpenShift console. The wording is a working draft;
> the module's published text is the lab author's.

In [ ]:
import labnb as lab
%cd ../agent

## Start from module 4

This puts the agent in the state module 4 starts from. Any edits you made are copied to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
!python reset.py 4

## 1. A clean-looking run

Portland is not one of Kestrel's depots.

In [ ]:
!python agent.py --q m4
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))

Fluent, authoritative, and reporting no problem. Now check the evidence rather than the answer.

In [ ]:
lab.trace_link()
lab.tool_results()

The tool was called, returned quickly, and the run ended cleanly on `stop`. What the tool actually
returned is a valid object with an empty results list and no sign that anything went wrong.

**Your read:** what else could have produced exactly that response? Open `run_tool` in
[`tools.py`](../agent/tools.py) and see what it does with an exception, and what `check_stock` does
with a depot it doesn't recognise.

```mermaid
flowchart TB
  subgraph Before["Before: four outcomes, one reply"]
    B1["nothing matched"] --> E1["results: [] (empty)"]
    B2["unknown depot"] --> E1
    B3["backend unreachable"] --> E1
    B4["bad arguments"] --> E1
  end
  subgraph After["After: the error contract"]
    A1["nothing matched"] --> C1["no_results"]
    A2["unknown depot"] --> C2["bad_request, naming the real depots"]
    A3["backend unreachable"] --> C3["upstream_unavailable, retryable"]
    A4["found it"] --> C4["ok, with the data"]
  end
  Before ~~~ After
```

## 2. Give failure a shape

Apply the supplied error contract:

In [ ]:
!cp solutions/module-04/tools.py solutions/module-04/agent.py .
!git diff --stat -- tools.py agent.py

Every tool return now carries a `category` (`ok`, `no_results`, `bad_request`,
`upstream_unavailable`), whether a retry could help, and a detail the model can use. `check_stock`
rejects an unknown depot as `bad_request`.

## 3. Run it again

In [ ]:
!python agent.py --q m4
lab.tool_results()
lab.trace_link()

**This is the module's success signal:** a `bad_request` that names the real depots, instead of a
confident answer. Two more, to see the other categories: a real depot with no stock, and a question
the data does cover.

In [ ]:
!python agent.py "Is the K-600 high-pressure sensor available at the Denver depot?"
!python agent.py --q m3
lab.evidence(columns=("question", "ended by", "tools run", "answer"), last=3)

The **tools run** column shows each call's category. In each trace, the tool span's outputs carry the
same `category`, `retryable` and `detail`.